In [3]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph
from typing import Annotated
from operator import add

graph_builder = StateGraph(Annotated[int, add])

graph_builder.add_node("get_one", lambda state: 1)
graph_builder.set_entry_point("get_one")
graph_builder.set_finish_point("get_one")

graph = graph_builder.compile(checkpointer=InMemorySaver())

result1 = graph.invoke(0, {"configurable": {"thread_id": "user1"}})
print(f"첫 번째 호출 결과: {result1}\n")
result2 = graph.invoke(0, {"configurable": {"thread_id": "user1"}})
print(f"두 번째 호출 결과: {result2}\n")
result3 = graph.invoke(0, {"configurable": {"thread_id": "user1"}})
print(f"세 번째 호출 결과: {result3}\n")

첫 번째 호출 결과: 1

두 번째 호출 결과: 2

세 번째 호출 결과: 3



In [4]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph
from langchain_openai import ChatOpenAI

class State(TypedDict):
    messages: Annotated[list, add_messages]
    
llm = ChatOpenAI(model="gpt-4o")
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.set_entry_point("chatbot")
graph_builder.set_finish_point("chatbot")
checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [5]:
graph.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요! 제 이름은 xx입니다. "}]},
    {"configurable": {"thread_id": "1"}}
)

{'messages': [HumanMessage(content='안녕하세요! 제 이름은 xx입니다. ', additional_kwargs={}, response_metadata={}, id='7c6cb439-f4ae-4cbd-961a-d30824dc6dce'),
  AIMessage(content='안녕하세요, xx님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 17, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_09f36310b5', 'id': 'chatcmpl-EdUb92VIEnK5WcpBxdqdYHlmInKok', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17bf6-8ca1-7a32-ae47-66db0d210585-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 19, 'total_tokens': 36, 'input_token_details': {'audio': 0

In [6]:
graph.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요! 제 이름이 뭐라고요?"}]},
    {"configurable": {"thread_id": "1"}}
)

{'messages': [HumanMessage(content='안녕하세요! 제 이름은 xx입니다. ', additional_kwargs={}, response_metadata={}, id='7c6cb439-f4ae-4cbd-961a-d30824dc6dce'),
  AIMessage(content='안녕하세요, xx님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 17, 'total_tokens': 36, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_09f36310b5', 'id': 'chatcmpl-EdUb92VIEnK5WcpBxdqdYHlmInKok', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a17bf6-8ca1-7a32-ae47-66db0d210585-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 19, 'total_tokens': 36, 'input_token_details': {'audio': 0

In [10]:
from langchain_core.messages.utils import (
    trim_messages,
    count_tokens_approximately
)
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver

model = init_chat_model("openai:gpt-4o")

def call_model(state: MessagesState):
    messages = trim_messages(
        state["messages"],
        strategy="last",
        token_counter=count_tokens_approximately,
        max_tokens=128,
        start_on="human",
        end_on=("human", "tool")
    )
    response = model.invoke(messages)
    return {"messages": [response]}

graph_builder = StateGraph(MessagesState)
graph_builder.add_node(call_model)
graph_builder.add_edge(START, "call_model")

checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [11]:
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": "안녕하세요, 저는 AI를 공부하는 학생입니다."}, config)
graph.invoke({"messages": "앞으로 AI를 사용할 때 주의해야 할 점을 조사하고 있어요."}, config)
graph.invoke({"messages": "또 어떤 주제에 대해 공부하면 좋을까요?"}, config)
final_response = graph.invoke({"messages": "제가 뭘 공부하고 있나요?"}, config)

final_response["messages"][-1].pretty_print()

================================== Ai Message ==================================

죄송하지만, 현재 무엇을 공부하고 있는지 알 수 없습니다. 공부하고 있는 주제나 과목을 말씀해주시면 그에 대해 정보를 제공하거나 도움을 드릴 수 있습니다. 어떤 것을 공부하고 계신가요?


In [14]:
from langchain_core.messages import HumanMessage, RemoveMessage
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, START, MessagesState, END
from langgraph.checkpoint.memory import InMemorySaver

class State(MessagesState):
    summary: str
    
model = init_chat_model("openai:gpt-4o")

def summarize_conversation(state: State):
    summary = state.get("summary", "")
    
    if summary:
        summary_message = (
            f"지금까지 대화 요약: {summary}\n\n"
            "위의 새로운 메시지를 고려하여 요약을 확장해주세요:"
        )
    else:
        summary_message = "위 대화 내용을 요약해주세요:"
        
    messages = state["messages"] + [HumanMessage(content=summary_message)]
    response = model.invoke(messages)
    
    # 최근 메시지 2개만 남기기
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    return {"summary": response.content, "messages": delete_messages}

In [15]:
def call_model(state: State):
    summary = state.get("summary", "")
    if summary:
        system_message = f"이전 대화 요약: {summary}"
        messages = [{"role": "system", "content": system_message}] + state["messages"]
    else:
        messages = state["messages"]
        
    response = model.invoke(messages)
    return {"messages": [response]}

def should_continue(state: State):
    messages = state["messages"]
    if len(messages) > 6:
        return "summarize_conversation"
    return "model"

graph_builder = StateGraph(State)
graph_builder.add_node("call_model", call_model)
graph_builder.add_node("summarize_conversation", summarize_conversation)

graph_builder.add_edge("summarize_conversation", "call_model")
graph_builder.add_conditional_edges(
    START,
    should_continue,
    {
        "model": "call_model",
        "summarize_conversation": "summarize_conversation"
    }
)
graph_builder.add_edge("call_model", END)

checkpointer = InMemorySaver()
graph = graph_builder.compile(checkpointer=checkpointer)

In [16]:
config = {"configurable": {"thread_id": "1"}}
graph.invoke({"messages": "안녕하세요, 저는 AI를 공부하는 학생입니다."}, config)
graph.invoke({"messages": "앞으로 AI를 사용할 때 주의해야 할 점을 조사하고 있어요."}, config)
graph.invoke({"messages": "또 어떤 주제에 대해 공부하면 좋을까요?"}, config)
final_response = graph.invoke({"messages": "제가 �뭘 공부하고 있나요?"}, config)

if final_response.get('summary'):
    print(f"요약: {final_response['summary']}\n")
final_response["messages"][-1].pretty_print()

요약: 당신은 AI를 공부하는 학생으로, AI를 사용할 때 주의해야 할 점을 조사하고 있습니다. 제가 AI 사용 시 주의사항에 대해 설명드렸고, 추가로 공부할만한 AI 관련 주제들을 제안해드렸습니다.

================================== Ai Message ==================================

이전 대화에서 AI를 공부하는 학생이라고 말씀하셨습니다. AI와 관련된 주제들, 특히 AI 사용 시 주의해야 할 점을 조사하고 있는 것으로 이해했습니다. 혹시 다른 구체적인 주제가 있거나 더 알고 싶으신 내용이 있으면 말씀해 주세요!
